# img2txt: дерматоскопический пайплайн

**Пайплайн из 4 шагов, каждый принимает и возвращает DataFrame:**

1. **Извлечение признаков** — сегментация + 60+ признаков (цвет, форма, граница, текстура)
2. **Бакетирование** — числовые признаки -> категориальные метки
3. **Ранжирование** — нейросеть выбирает топ-10 важных признаков
4. **Генерация текста** — Mistral-7B генерирует клиническое описание на русском

---

# Часть 1. Обучение / исследование (batch)

In [ ]:
import pandas as pd
import torch

from extraction.feature_extraction_batch import extract_features_batch, images_to_df
from analysis.feature_bucketing_batch import bucket_features_batch, get_label_statistics
from importance.importance_inference import rank_features_batch
from generation.description_inference import generate_descriptions_batch
from generation.classification_types import ClassificationResult, Structure, FeatureType

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Пути (Kaggle)
IMAGE_DIR = "/kaggle/input/datasets/mihailodin1/all-image-skin"
YOLO_WEIGHTS = "/kaggle/input/weight-mask/weight/mask_builder_yolo.pt"
UNET_WEIGHTS = "/kaggle/input/weight-mask/weight/mask_builder_unet.pth"
IMPORTANCE_CHECKPOINT = "importance_checkpoints/best.pt"
FEATURES_CSV = "/kaggle/input/datasets/mihailodin1/features-img2txt/features_dataset.csv"

## Шаг 1. Извлечение признаков

In [ ]:
# Вариант A: извлечь признаки из директории с изображениями
df = images_to_df(IMAGE_DIR)
df = extract_features_batch(df, yolo_weights=YOLO_WEIGHTS, unet_weights=UNET_WEIGHTS)
df.to_csv("features_dataset.csv", index=False)
print(f"Извлечено: {len(df)} изображений, успешно: {(df['status'] == 'success').sum()}")

In [ ]:
# Вариант B: загрузить ранее вычисленные признаки
df = pd.read_csv(FEATURES_CSV)
print(f"Загружено: {len(df)} изображений")

## Шаг 2. Бакетирование признаков

In [ ]:
df = bucket_features_batch(df)
print(f"Добавлены колонки: labels, features_organized, labels_json")

In [ ]:
# Статистика по меткам
stats = get_label_statistics(df)
if not stats.empty:
    print(stats.to_string())

## Шаг 3. Ранжирование важных признаков

### 3a. Генерация псевдо-меток (для обучения модели)

In [ ]:
from importance.pseudo_importance import add_pseudo_important_labels

df = add_pseudo_important_labels(
    df,
    labels_col="labels",
    features_col="features_json",
    top_k=15,
    mode="combined",
    z_weight=1.5,
    prior_weight=0.8,
)
df["important_labels"] = df["pseudo_important_labels"]

print("Пример псевдо-меток:")
print(df["pseudo_important_labels"].iloc[0])

### 3b. Обучение модели важности

In [ ]:
from importance.train_importance import train_importance

# Сохранить датасет для обучения
train_csv = "features_dataset_for_importance.csv"
df.to_csv(train_csv, index=False)

# Обучение
best_score = train_importance(
    data_csv=train_csv,
    image_dir=IMAGE_DIR,
    backbone="efficientnet_b0",
    epochs=70,
    batch_size=64,
    lr=1e-4,
    out_dir="importance_checkpoints",
)
print(f"Best validation score: {best_score:.4f}")

### 3c. Ранжирование обученной моделью (batch)

In [ ]:
df = rank_features_batch(df, importance_model_path=IMPORTANCE_CHECKPOINT, device=device)

print("Пример important_labels:")
print(df["important_labels"].iloc[0])

## Шаг 4. Генерация клинических описаний (Mistral-7B)

Первый вызов загружает модель (~14GB). Последующие используют кэш.

In [ ]:
df = generate_descriptions_batch(df, device=device)

print("Пример описания:")
print(df["description"].iloc[0])

In [ ]:
# Сохранение результатов
output_csv = "features_with_descriptions.csv"
df.to_csv(output_csv, index=False)
print(f"Сохранено: {len(df)} строк в {output_csv}")

---

# Часть 2. Инференс (single image)

Тот же пайплайн, но для одного изображения — однострочный DataFrame.

In [ ]:
image_path = "/path/to/lesion.jpg"

# Шаг 1: извлечение признаков
df_single = pd.DataFrame([{"image_path": image_path}])
df_single = extract_features_batch(df_single, yolo_weights=YOLO_WEIGHTS, unet_weights=UNET_WEIGHTS, verbose=False)

# Шаг 2: бакетирование
df_single = bucket_features_batch(df_single, verbose=False)

# Шаг 3: ранжирование
df_single = rank_features_batch(df_single, importance_model_path=IMPORTANCE_CHECKPOINT, device=device, verbose=False)

# Шаг 4: генерация описания (с классификацией)
df_single["classification"] = ClassificationResult(
    feature_type=FeatureType.SINGLE,
    structure=Structure.GLOBULES,
    properties=["однородный"],
    final_class="Меланома",
)
df_single = generate_descriptions_batch(df_single, classification_col="classification", device=device, verbose=False)

print(df_single.iloc[0]["description"])